# 08. Using Pre-Trained LLMs: Quantization & Device Mapping

Serving modern 7B–70B open LLMs locally requires memory optimization techniques:

### Production Serving Optimizations:
- **`device_map='auto'`** (via Accelerate): Automatically shards model layers across available GPUs and CPU RAM.
- **BitsAndBytes Quantization**: Converts 16-bit FP16 weights into 8-bit (LLM.int8()) or 4-bit NormalFloat (NF4) representations, cutting VRAM by ~75% with minimal perplexity degradation!

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "gpt2" # Lightweight demonstration model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

prompt = "Artificial Intelligence will transform healthcare by"
inputs = tokenizer(prompt, return_tensors="pt")

# Generate text with top-p sampling
with torch.no_grad():
    outputs = model.generate(
        **inputs, 
        max_new_tokens=25,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Generated Completion:\n", generated_text)

## 🎯 Exercises & Practice
1. Calculate how much GPU VRAM is required to load a 7-Billion parameter model in: 16-bit FP16 vs 4-bit NF4 quantization.

In [ ]:
# Exercise Solution
params = 7e9
fp16_gb = (params * 2) / (1024**3)
nf4_gb = (params * 0.5) / (1024**3)
print(f"FP16 (16-bit): {fp16_gb:.2f} GB (Requires A100 or 24GB RTX 3090/4090)")
print(f"NF4  (4-bit):  {nf4_gb:.2f} GB (Comfortably fits on a consumer 8GB GPU!)")